# Study 940 — The Turnover Budget ⏱️

**One sleeve, four speeds. What does rebalancing faster actually cost — and is there
anything there to pay for it?**

The folk theorem: rebalance more often and you track a decaying signal better, but you pay
in turnover, so there is an optimal frequency somewhere. It is almost never *priced*. Here
we price it. The sleeve is fixed — rank the eleven **Select Sector SPDRs** on their **12-1**
total return, long the top 3, short the bottom 3, dollar-neutral at 1.0 gross. The only
thing that changes is the rebalance clock: **daily / weekly / monthly / quarterly**.

The deliverable is not a Sharpe. It is each speed's **break-even cost per unit of traded
notional** — how many basis points of friction that clock can afford before its net excess
return hits zero.

Tape: 1999-12-23 → 2026-06-30 (6,668 days), daily **total-return** closes.
One execution lag (signal through day *t*, weights effective *t+1*, trade charged *t+1*);
weights drift between rebalances; costs are one-way × NAV; the short leg pays borrow.

*Real numbers below are frozen from `docs/results.md` (Fingerprint `7af2719e8031`, as-of
2026-06-30). The only live cells run a small **synthetic** panel and say so.*


## 1. The bill, in one number

A book that re-ranks itself **every day** trades about **31 times its own net asset value each year**. The same book on a **monthly** clock trades **5.8 times**. That is a 5.4× difference in the bill — before anyone asks what the extra trading bought.

In [1]:
R = {'labels': ['daily', 'weekly', 'monthly', 'quarterly'], 'n_rebal': [6668, 1384, 318, 106], 'turnover': [31.27, 13.44, 5.81, 3.27], 'turn_per_rebal': [0.1241, 0.2569, 0.4828, 0.8143]}
print('speed      rebalances   turnover/yr   per rebalance')
for i, lab in enumerate(R['labels']):
    print('%-10s %10d %12.1fx %14.1f%%'
          % (lab, R['n_rebal'][i], R['turnover'][i], R['turn_per_rebal'][i]*100))

speed      rebalances   turnover/yr   per rebalance
daily            6668         31.3x           12.4%
weekly           1384         13.4x           25.7%
monthly           318          5.8x           48.3%
quarterly         106          3.3x           81.4%


## 2. What the extra trading bought: almost nothing

Before any cost, the daily clock earned **0.98%** a year and the monthly clock **0.79%**. That is a gross Sharpe gap of **+0.012** — and it is not distinguishable from luck (*t* = +0.18). The quarterly clock earned **0.10%**: nothing at all.

So the fast clock pays 5.4× the bill for a rounding error.

In [2]:
R = {'labels': ['daily', 'weekly', 'monthly', 'quarterly'], 'ret_gross': [0.98, 0.85, 0.79, 0.1], 'sharpe_gross': [0.121, 0.105, 0.099, 0.013], 't_gross': [0.67, 0.59, 0.55, 0.07]}
print('speed        gross return/yr   gross Sharpe   HAC t')
for i, lab in enumerate(R['labels']):
    print('%-10s %14.2f%% %14.3f %8.2f'
          % (lab, R['ret_gross'][i], R['sharpe_gross'][i], R['t_gross'][i]))
print('\nbest gross t across all four speeds: %+.2f  -> nothing here is real'
      % max(R['t_gross']))

speed        gross return/yr   gross Sharpe   HAC t
daily                0.98%          0.121     0.67
weekly               0.85%          0.105     0.59
monthly              0.79%          0.099     0.55
quarterly            0.10%          0.013     0.07

best gross t across all four speeds: +0.67  -> nothing here is real


## 3. The budget — and why it is worthless

The **break-even cost** is the honest deliverable: how many basis points per unit of traded notional each speed can afford before it earns exactly zero.

| speed | break-even |
|---|--:|
| daily | **2.5 bps** |
| weekly | **4.8 bps** |
| monthly | **10.1 bps** |
| quarterly | **-3.2 bps** (negative — it loses money before a single trade is charged) |

Read that as a price of admission: a daily sector-momentum book needs its costs under two and a half basis points per unit traded, forever, to break even. But the budget is only as trustworthy as the return funding it — and that return has a *t* of at most +0.67. **This is a budget for zero.**

> 🔬 **For the quants** — break-even is the zero of `mean(excess) = gross_mean − borrow − c × 1e-4 × mean(traded)` solved for `c`, with borrow still charged (it is a holding cost, not a trading cost). Turnover here is traded notional `Σ|w_new − w_drifted|` as a fraction of NAV, so `c` is in bps *per unit traded*, not per rebalance.

## 4. The ranking of speeds is decided by your broker, not by the tape

At zero cost the daily clock is the best arm. By **one basis point** it is already third, and by 2.5 bps it is the **worst**. The whole inversion happens **below the first basis point** of execution cost — a standard nobody clears on a book that trades 31× NAV a year.

Which is the real lesson: 'what is the optimal rebalance frequency' is not a question about markets. It is a question about your fill.

In [3]:
R = {'labels': ['daily', 'weekly', 'monthly', 'quarterly'], 'cost_grid': [0.0, 1.0, 2.5, 5.0, 10.0, 25.0], 'cost_surface': [[0.097, 0.08, 0.073, -0.014], [0.058, 0.063, 0.066, -0.018], [0.0, 0.039, 0.055, -0.024], [-0.096, -0.003, 0.037, -0.034], [-0.288, -0.086, 0.001, -0.055], [-0.862, -0.334, -0.109, -0.118]]}
print('cost/unit turnover |' + ''.join('%10s' % l for l in R['labels']))
for c, row in zip(R['cost_grid'], R['cost_surface']):
    print('%14.1f bps |' % c + ''.join('%+10.3f' % v for v in row))
print('\nnet excess-of-cash Sharpe. daily leads at 0 bps and trails by 2.5 bps.')

cost/unit turnover |     daily    weekly   monthly quarterly
           0.0 bps |    +0.097    +0.080    +0.073    -0.014
           1.0 bps |    +0.058    +0.063    +0.066    -0.018
           2.5 bps |    +0.000    +0.039    +0.055    -0.024
           5.0 bps |    -0.096    -0.003    +0.037    -0.034
          10.0 bps |    -0.288    -0.086    +0.001    -0.055
          25.0 bps |    -0.862    -0.334    -0.109    -0.118

net excess-of-cash Sharpe. daily leads at 0 bps and trails by 2.5 bps.


## 5. The trap this study exists to name

Drop the short leg and run the same ranking **long-only** — top 3 sectors, fully invested — and suddenly every speed looks significant against cash (*t* ≈ +2.6 to +2.6, Sharpe ~0.53). It would be very easy to publish that.

It is equity beta. Cash is the wrong yardstick for a book that is 100% in equities all the time. The right one is the **same eleven sectors, equal-weighted, on the same clock, paying the same commission** — a control with no view at all. Against that, momentum selection adds **+0.67%/yr** before trading costs at the daily clock (*t* = +0.35) and **-0.31%/yr** at quarterly (*t* = -0.17) — nothing, at any speed.

Net of costs it is worse, and for the reason this whole study is about: the sleeve turns over **27× NAV a year** to the control's **1.5×**, so it hands back -0.62%/yr at daily. Selection bought nothing and the trading bill was real.

> ⚠️ **How this study got it wrong first.** The original cut of this table raced the costed momentum sleeve against a *frictionless*, daily-rebalanced equal-weight average. That one-sided ledger charged one horse and not the other, and printed a −0.7%/yr 'selection shortfall' at the daily clock that was, to the basis point, the strategy's own commission. Same-clock, same-cost, or the race means nothing.

In [4]:
R = {'labels': ['daily', 'weekly', 'monthly', 'quarterly'], 'lo_t_cash': [2.56, 2.65, 2.62, 2.62], 'lo_sharpe_ew': [0.556, 0.555, 0.551, 0.584], 'lo_alpha': [-0.62, -0.19, -0.15, -0.44], 'lo_t_ew': [-0.33, -0.1, -0.08, -0.24], 'lo_alpha_gross': [0.67, 0.4, 0.09, -0.31], 'lo_t_ew_gross': [0.35, 0.21, 0.05, -0.17], 'lo_turn': [27.2, 12.44, 5.21, 2.82], 'lo_turn_ew': [1.49, 0.72, 0.37, 0.24]}
hdr = 'speed      vs cash (t)   EW-11   alpha GROSS (t)     alpha NET (t)   turn  turnEW'
print(hdr); print('-'*len(hdr))
for i, lab in enumerate(R['labels']):
    print('%-9s %10.2f %8.3f %9.2f%% (%+5.2f) %9.2f%% (%+5.2f) %6.1fx %6.1fx'
          % (lab, R['lo_t_cash'][i], R['lo_sharpe_ew'][i],
             R['lo_alpha_gross'][i], R['lo_t_ew_gross'][i],
             R['lo_alpha'][i], R['lo_t_ew'][i],
             R['lo_turn'][i], R['lo_turn_ew'][i]))
print('\nalpha is per year vs the equal-weight control. Nothing clears |t| = 2.')

speed      vs cash (t)   EW-11   alpha GROSS (t)     alpha NET (t)   turn  turnEW
---------------------------------------------------------------------------------
daily           2.56    0.556      0.67% (+0.35)     -0.62% (-0.33)   27.2x    1.5x
weekly          2.65    0.555      0.40% (+0.21)     -0.19% (-0.10)   12.4x    0.7x
monthly         2.62    0.551      0.09% (+0.05)     -0.15% (-0.08)    5.2x    0.4x
quarterly       2.62    0.584     -0.31% (-0.17)     -0.44% (-0.24)    2.8x    0.2x

alpha is per year vs the equal-weight control. Nothing clears |t| = 2.


## 6. Live check — the machinery works when there is something to find

**This cell is synthetic, not the real tape.** We plant a cross-section whose expected returns really do persist for about a quarter, and check that the ladder recovers the mechanism: on a world with genuine decaying momentum the faster clock *should* earn more gross. Then we switch the signal off and check the ladder goes quiet. If both hold, the flat real-tape result is a fact about sector momentum rather than a broken backtest.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
from turnover_budget import data, strategy as st
p1, c1, _ = data.synthetic_panel(n_assets=8, n_years=12, signal_strength=1.0, seed=940)
p0, c0, _ = data.synthetic_panel(n_assets=8, n_years=12, signal_strength=0.0, seed=940)
d1 = st.synthetic_detect(p1, c1, top_k=2)
d0 = st.synthetic_detect(p0, c0, top_k=2)
print('SYNTHETIC panel with planted momentum -- gross return per year:')
for f in ('D','W','M','Q'):
    print('   %-9s %+7.2f%%  (t %+6.2f)' % (st.FREQ_LABEL[f],
          d1['ann_return_gross'][f]*100, d1['t_gross'][f]))
print('   -> faster clock earns more: the mechanism is recovered\n')
print('SYNTHETIC null panel (no momentum planted) -- gross return per year:')
for f in ('D','W','M','Q'):
    print('   %-9s %+7.2f%%  (t %+6.2f)' % (st.FREQ_LABEL[f],
          d0['ann_return_gross'][f]*100, d0['t_gross'][f]))
print('   -> silent, as it must be')

SYNTHETIC panel with planted momentum -- gross return per year:
   daily      +34.59%  (t +11.15)
   weekly     +32.91%  (t +10.68)
   monthly    +28.49%  (t  +9.37)
   quarterly  +22.90%  (t  +7.55)
   -> faster clock earns more: the mechanism is recovered

SYNTHETIC null panel (no momentum planted) -- gross return per year:
   daily       -3.55%  (t  -1.67)
   weekly      -3.23%  (t  -1.51)
   monthly     +0.36%  (t  +0.17)
   quarterly   -0.57%  (t  -0.26)
   -> silent, as it must be


## Verdict

- **Signal — None.** Over 26.5 years the sleeve's best gross HAC *t* at any speed is **+0.67**. Every bootstrap Sharpe interval straddles zero; no speed-versus-speed race clears |*t*| = 2. The long-only arm's apparent significance is equity beta — measured against the same eleven sectors equal-weighted on the same clock and paying the same costs, its selection alpha is indistinguishable from zero at every speed (largest |*t*| = 0.35).
- **Tradability — Mirage.** The budget is 2.5 bps daily, 4.8 weekly, 10.1 monthly and negative quarterly — a budget for a return that is statistically zero. At a realistic 5 bps plus 40 bps borrow, three of four speeds are net-negative and the fourth is indistinguishable from cash.
- **What survives.** The ladder itself. A daily clock on an eleven-name cross-section trades 31× NAV a year and therefore needs roughly **five times** the gross edge of a monthly clock just to stand still. That arithmetic transfers to any sleeve you care about — ask of your signal whether its *gross* alpha clears the break-even column. This one does not.